# Milestone 2.4 — Kaggle GPU execution

This notebook orchestrates the existing Milestone 2.3 APIs. Select Kaggle GPU T4 x2. Both devices are inventoried, but all model work is explicitly restricted to `cuda:0`. Full training and smoke tests are opt-in and disabled by default.

Model/tokenizer revision `12040accade4e8a0f71eabdb258fecc2e7e948be` is the locally tested immutable revision, not an independently approved production revision. Baseline/canonical results are a whole-system comparison; they do not isolate byte-feature effects. The held-out test split is not loaded or evaluated.

In [ ]:
# Safe defaults: Run All will not start training.
ALLOW_PINNED_ASSET_DOWNLOAD = False
RUN_CUDA_SMOKE_TESTS = False
RUN_FULL_BASELINE_TRAINING = False
RUN_FULL_CANONICAL_TRAINING = False
REQUESTED_MIXED_PRECISION = 'fp16'  # request only; resolve and record effective mode
MODEL_ID = TOKENIZER_ID = 'distilbert-base-uncased'
MODEL_REVISION = TOKENIZER_REVISION = '12040accade4e8a0f71eabdb258fecc2e7e948be'
DEVICE_NAME = 'cuda:0'
SEED, MICROBATCH_SIZE, GRADIENT_ACCUMULATION_STEPS = 42, 2, 8
EFFECTIVE_BATCH_SIZE = MICROBATCH_SIZE * GRADIENT_ACCUMULATION_STEPS
EPOCHS, MAX_EXPANDED_LENGTH, MAX_BYTE_SPAN_LENGTH = 3, 128, 512
LEARNING_RATE, WEIGHT_DECAY, WARMUP_RATIO, GRADIENT_CLIP_NORM = 2e-5, .01, .1, 1.0
VALIDATION_INTERVAL_STEPS, NUM_WORKERS = 0, 0
print({'device': DEVICE_NAME, 'effective_batch_size': EFFECTIVE_BATCH_SIZE, 'automatic_training': False})

## 1. Environment and GPU detection
Inventory both T4s; select only `cuda:0`.

In [ ]:
import json, os, platform, subprocess, sys, time
from pathlib import Path
import torch, uuid
SESSION_ID = time.strftime('%Y%m%d-%H%M%S') + '-' + uuid.uuid4().hex[:8]
ARTIFACT_ROOT = Path('/kaggle/working') / f'nlp-defense-m24-{SESSION_ID}'
SPLIT_DIR = Path('/kaggle/working') / f'nlp-defense-m24-splits-{SESSION_ID}'
assert sys.version_info >= (3, 10), f'Python 3.10+ required, found {platform.python_version()}'
assert torch.cuda.is_available(), 'CUDA unavailable: select GPU T4 x2 in Kaggle settings and restart.'
gpus = []
for i in range(torch.cuda.device_count()):
 p = torch.cuda.get_device_properties(i); gpus.append({'index':i,'name':p.name,'memory_bytes':p.total_memory,'memory_gib':round(p.total_memory/2**30,3),'capability':[p.major,p.minor]})
assert gpus, 'No CUDA devices found.'
device = torch.device('cuda:0'); torch.cuda.set_device(device)
runtime_report = {'python':platform.python_version(),'torch':torch.__version__,'torch_cuda_runtime':torch.version.cuda,'cuda_device_count':torch.cuda.device_count(),'gpus':gpus,'selected_device':str(device),'selected_name':torch.cuda.get_device_name(0),'selected_capability':torch.cuda.get_device_capability(0),'selected_memory_bytes':torch.cuda.get_device_properties(0).total_memory}
try: runtime_report['nvidia_smi'] = subprocess.run(['nvidia-smi','-L'],check=True,capture_output=True,text=True).stdout
except (OSError, subprocess.CalledProcessError) as exc: runtime_report['nvidia_smi_note'] = str(exc)
print(json.dumps(runtime_report,indent=2)); assert str(device) == 'cuda:0'


## 2. Dependency compatibility and CUDA probe
Do not install or upgrade packages in the Kaggle kernel. A version mismatch stops the workflow instead of replacing Kaggle's CUDA-enabled PyTorch build.

In [ ]:
from importlib.metadata import PackageNotFoundError, version
import transformers, tokenizers
expected = {'torch':'2.10.0','transformers':'5.2.0','tokenizers':'0.22.2'}
observed = {'torch':torch.__version__.split('+')[0],'transformers':transformers.__version__,'tokenizers':tokenizers.__version__}
for name, wanted in expected.items():
 if observed[name] != wanted: raise RuntimeError(f'{name} expected {wanted}, found {observed[name]}; review compatibility, do not install over Kaggle CUDA image')
extra = {}
for name in ('numpy','pandas','scikit-learn','safetensors','huggingface-hub'):
 try: extra[name] = version(name)
 except PackageNotFoundError: extra[name] = 'missing'
x = torch.ones((32,32),device=device); y = x @ x; assert y.device == device and torch.isfinite(y).all().item(); del x,y
print({'expected_core':expected,'observed_core':observed,'other_versions':extra,'torch_cuda_build':torch.version.cuda,'selected_gpu':torch.cuda.get_device_name(0),'cuda0_tensor_probe':'passed'})

## 3. Repository setup
Prefer an attached source dataset at `/kaggle/input/nlp-defense-source`; otherwise explicitly clone into `/kaggle/working` when Kaggle Internet is enabled. Never write under `/kaggle/input`.

In [ ]:
import subprocess, sys
INPUT_REPO = Path('/kaggle/input/nlp-defense-source'); WORK_REPO = Path('/kaggle/working/nlp-defense'); REPOSITORY_REF = 'main'; REPOSITORY_COMMIT = 'efa0a1ac2aa5dfbccaa8e14b17a193b71e0713e2'
if (INPUT_REPO/'src'/'training.py').is_file(): REPO_ROOT = INPUT_REPO
elif (WORK_REPO/'src'/'training.py').is_file(): REPO_ROOT = WORK_REPO
else:
 if os.environ.get('KAGGLE_KERNEL_INTERNET','').lower() not in {'1','true'}: raise FileNotFoundError('Attach repository source or enable Kaggle Internet and rerun this clone cell.')
 subprocess.run(['git','clone','--depth','1','--branch',REPOSITORY_REF,'https://github.com/Suhaniverma2004/nlp-defense.git',str(WORK_REPO)],check=True)
 subprocess.run(['git','-C',str(WORK_REPO),'fetch','--depth','1','origin',REPOSITORY_COMMIT],check=True)
 subprocess.run(['git','-C',str(WORK_REPO),'checkout','--detach',REPOSITORY_COMMIT],check=True); REPO_ROOT = WORK_REPO
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path: sys.path.insert(0,str(REPO_ROOT))
expected_source_commit = 'efa0a1ac2aa5dfbccaa8e14b17a193b71e0713e2'
try:
 source_commit = subprocess.run(['git','-C',str(REPO_ROOT),'rev-parse','HEAD'],check=True,capture_output=True,text=True).stdout.strip()
except subprocess.CalledProcessError:
 marker = REPO_ROOT/'MILESTONE_2_4_SOURCE_COMMIT.txt'
 source_commit = marker.read_text(encoding='utf-8').strip() if marker.is_file() else None
if source_commit != expected_source_commit:
 raise RuntimeError(f'Repository source must identify inspected commit {expected_source_commit}; found {source_commit!r}. For a Kaggle input dataset without .git, include MILESTONE_2_4_SOURCE_COMMIT.txt containing that SHA.')
runtime_report['repository_commit'] = source_commit
print('Repository:',REPO_ROOT,'commit:',source_commit)

## 4. Dataset provenance and split verification
Attach local SST-2 source CSVs. Existing split preparation creates the reproducible train/validation/held-out partitions in `/kaggle/working`. This notebook loads only train and validation rows; it never loads or evaluates the held-out split.

In [ ]:
from src.sst2_data import prepare_reproducible_splits, load_split
def first_existing(paths):
 for p in paths:
  p = Path(p)
  if p.is_file(): return p
 return None
TRAIN_SOURCE = first_existing([REPO_ROOT/'data/sst2/sst2_train.csv','/kaggle/input/sst2/sst2_train.csv','/kaggle/input/glue-sst2/train.csv'])
VALIDATION_SOURCE = first_existing([REPO_ROOT/'data/sst2/sst2_validation.csv','/kaggle/input/sst2/sst2_validation.csv','/kaggle/input/glue-sst2/validation.csv'])
if TRAIN_SOURCE is None or VALIDATION_SOURCE is None: raise FileNotFoundError('Attach local SST-2 train and original-validation CSVs; no data download is attempted.')
SPLIT_DIR.mkdir(parents=True,exist_ok=True)
if not (SPLIT_DIR/'manifest.json').exists():
 if any(SPLIT_DIR.iterdir()): raise FileExistsError(f'Non-empty split path without manifest: {SPLIT_DIR}')
 prepare_reproducible_splits(TRAIN_SOURCE,VALIDATION_SOURCE,SPLIT_DIR,seed=SEED)
train_frame, manifest = load_split(SPLIT_DIR,'train'); validation_frame, validation_manifest = load_split(SPLIT_DIR,'validation')
if manifest != validation_manifest or set(train_frame.sentence)&set(validation_frame.sentence): raise ValueError('Split provenance mismatch or train/validation text overlap.')
print(json.dumps({'train_rows':len(train_frame),'validation_rows':len(validation_frame),'train_source_sha256':manifest['source']['train_sha256'],'original_validation_sha256':manifest['source']['original_validation_sha256'],'seed':manifest['seed'],'strategy':manifest['split_strategy'],'held_out_split_loaded':False},indent=2))

## 5. Model and tokenizer revision verification
Asset loading is local-cache-only unless the explicit switch at the top is changed. The requested SHA is locally tested, not production-approved.

In [ ]:
from transformers import AutoConfig, AutoTokenizer
from src.training import PINNED_REPRODUCIBILITY_REVISION
assert MODEL_REVISION == TOKENIZER_REVISION == PINNED_REPRODUCIBILITY_REVISION
local_only = not ALLOW_PINNED_ASSET_DOWNLOAD
tok = AutoTokenizer.from_pretrained(TOKENIZER_ID,revision=TOKENIZER_REVISION,use_fast=True,local_files_only=local_only)
model_config = AutoConfig.from_pretrained(MODEL_ID,revision=MODEL_REVISION,local_files_only=local_only)
if not tok.is_fast: raise RuntimeError('Fast tokenizer with offset support required.')
model_commit = getattr(model_config,'_commit_hash',None); tokenizer_commit = getattr(tok,'init_kwargs',{}).get('_commit_hash')
print({'requested_revision':MODEL_REVISION,'resolved_model_commit':model_commit,'resolved_tokenizer_commit':tokenizer_commit,'fast_tokenizer':tok.is_fast,'download_allowed':ALLOW_PINNED_ASSET_DOWNLOAD})
if model_commit != MODEL_REVISION: raise RuntimeError(f'Model commit metadata missing or mismatched: {model_commit!r}')
if tokenizer_commit != TOKENIZER_REVISION: raise RuntimeError(f'Tokenizer commit metadata missing or mismatched: {tokenizer_commit!r}')
del tok, model_config

## 6. Precision resolution and shared settings
FP16 is a request only. Record the effective resolved dtype and execution metadata before training.

In [ ]:
from src.training import TrainingConfig, load_pretrained_components, resolve_mixed_precision, execution_metadata
effective_dtype = resolve_mixed_precision(REQUESTED_MIXED_PRECISION,device)
if effective_dtype is None: effective_precision = 'none'
else:
 with torch.autocast(device_type='cuda',dtype=effective_dtype): probe = torch.ones((16,16),device=device) @ torch.ones((16,16),device=device)
 if not torch.isfinite(probe).all().item(): raise FloatingPointError('Autocast probe produced non-finite values.')
 effective_precision = str(effective_dtype).replace('torch.',''); del probe
SHARED = dict(model_id=MODEL_ID,model_revision=MODEL_REVISION,tokenizer_id=TOKENIZER_ID,tokenizer_revision=TOKENIZER_REVISION,max_expanded_length=MAX_EXPANDED_LENGTH,max_byte_span_length=MAX_BYTE_SPAN_LENGTH,batch_size=MICROBATCH_SIZE,gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,seed=SEED,epochs=EPOCHS,learning_rate=LEARNING_RATE,weight_decay=WEIGHT_DECAY,warmup_ratio=WARMUP_RATIO,gradient_clip_norm=GRADIENT_CLIP_NORM,mixed_precision=REQUESTED_MIXED_PRECISION,validation_interval_steps=VALIDATION_INTERVAL_STEPS,num_workers=NUM_WORKERS,device=DEVICE_NAME)
print({'requested_precision':REQUESTED_MIXED_PRECISION,'effective_precision':effective_precision,'execution':execution_metadata(device,effective_dtype),'shared_settings':SHARED,'comparison':'whole-system only; byte-feature effect is not isolated'})

## 7. Bounded CUDA smoke, checkpoint reload, and epoch-boundary resume
Disabled by default. When explicitly enabled, both models use at most four training and four validation examples. Existing trainer/checkpoint APIs are used. The cell simulates a pointer-write interruption after snapshot publication, invokes snapshot recovery, resumes, reloads the best checkpoint offline, and runs inference.

In [ ]:
if RUN_CUDA_SMOKE_TESTS:
 from unittest.mock import patch
 from src.training import fit_model, ClassificationCollator
 from src.checkpointing import recover_latest_resume_checkpoint, load_checkpoint
 import pandas as pd
 smoke_train = train_frame.groupby('label',group_keys=False).head(2).head(4).reset_index(drop=True)
 smoke_valid = validation_frame.groupby('label',group_keys=False).head(2).head(4).reset_index(drop=True)
 if len(smoke_train)!=4 or len(smoke_valid)!=4 or set(smoke_train.sentence)&set(smoke_valid.sentence): raise ValueError('Need four disjoint rows in each smoke split.')
 provenance = {'kind':'bounded-smoke-subset','source_manifest':manifest,'train_ids':smoke_train.get('source_row_index',pd.Series(range(4))).tolist(),'validation_ids':smoke_valid.get('source_row_index',pd.Series(range(4))).tolist()}
 smoke_results = {}; smoke_root = ARTIFACT_ROOT/'smoke'
 for kind in ('baseline','canonical'):
  out=smoke_root/kind; out.mkdir(parents=True,exist_ok=True)
  if any(out.iterdir()): raise FileExistsError(f'Use fresh smoke directory: {out}')
  cfg=TrainingConfig(**{**SHARED,'epochs':2,'output_dir':str(out)})
  model,tokenizer=load_pretrained_components(cfg,model_kind=kind,local_files_only=not ALLOW_PINNED_ASSET_DOWNLOAD)
  collator=ClassificationCollator(tokenizer,cfg,kind); before=next(model.parameters()).detach().cpu().clone(); torch.cuda.reset_peak_memory_stats(device); started=time.perf_counter()
  with patch('src.checkpointing.write_latest_resume_pointer',side_effect=RuntimeError('simulated interruption')):
   try: fit_model(model,tokenizer,smoke_train,smoke_valid,cfg,model_kind=kind,checkpoint_root=out/'checkpoints',dataset_provenance=provenance,history_path=out/'training_history.json')
   except RuntimeError as exc:
    if 'simulated interruption' not in str(exc): raise
  snapshot=recover_latest_resume_checkpoint(out/'checkpoints')
  if snapshot is None: raise RuntimeError(f'No valid recovered snapshot for {kind}.')
  del model,tokenizer; torch.cuda.empty_cache()
  model2,tok2=load_pretrained_components(cfg,model_kind=kind,local_files_only=True)
  result=fit_model(model2,tok2,smoke_train,smoke_valid,cfg,model_kind=kind,checkpoint_root=out/'checkpoints',dataset_provenance=provenance,resume_checkpoint_dir=snapshot,history_path=out/'training_history.json')
  if not result.best_checkpoint_dir or not Path(result.best_checkpoint_dir).is_dir(): raise RuntimeError('Best checkpoint missing.')
  if torch.equal(before,next(result.model.parameters()).detach().cpu()): raise AssertionError('No optimizer update observed.')
  os.environ['HF_HUB_OFFLINE']='1'; os.environ['TRANSFORMERS_OFFLINE']='1'
  reloaded,rtok,meta=load_checkpoint(result.best_checkpoint_dir,expected_model_kind=kind,expected_model_id=MODEL_ID,expected_model_revision=MODEL_REVISION,expected_tokenizer_id=TOKENIZER_ID,expected_tokenizer_revision=TOKENIZER_REVISION,device=device)
  row=smoke_valid.iloc[0]; batch=collator([(row.sentence,int(row.label),0)]); inp=batch['inputs']
  if kind=='canonical': logits=reloaded(inp.to(device)).logits
  else: logits=reloaded(inp['input_ids'].to(device),inp['attention_mask'].to(device)).logits
  if logits.shape!=(1,2) or not torch.isfinite(logits).all(): raise FloatingPointError('Reloaded inference invalid.')
  if any(not torch.isfinite(torch.tensor(float(v))) for v in result.best_validation_metrics.values() if isinstance(v,(int,float))): raise FloatingPointError('Non-finite validation metrics.')
  smoke_results[kind]={'optimizer_update':True,'recovered_snapshot':str(snapshot),'validation':result.best_validation_metrics,'elapsed_seconds':time.perf_counter()-started,'peak_cuda_bytes':torch.cuda.max_memory_allocated(device),'effective_precision':result.effective_mixed_precision,'checkpoint_kind':meta['model_kind'],'logits_shape':list(logits.shape)}
  del model2,result,reloaded,rtok,logits; torch.cuda.empty_cache()
 print(json.dumps(smoke_results,indent=2,default=str))
else: print('Skipped by default; set RUN_CUDA_SMOKE_TESTS=True after checks.')

## 8. GPU memory and timing diagnostics
The smoke cell records elapsed wall time and peak allocated memory. These are engineering diagnostics, not benchmark results. On OOM, stop the cell, retain the last completed snapshot, restart from it, and reduce microbatch from 2 to 1 while increasing accumulation from 8 to 16 for both models. Keep effective batch size 16. Do not silently truncate inputs.

In [ ]:
if 'smoke_results' in globals():
 for name,r in smoke_results.items(): print(name,{k:r[k] for k in ('elapsed_seconds','peak_cuda_bytes','effective_precision')})
else: print('No timing sample has run.')

## 9. Baseline training — explicit opt-in
Uses `experiment.train_sst2`; training is never started automatically. Keep the complete output directory to resume from `checkpoints/latest_checkpoint.json` or the checkpoint root.

In [ ]:
BASELINE_OUTPUT=ARTIFACT_ROOT/'baseline'
def train_command(kind,out):
 return [sys.executable,'-m','experiment.train_sst2','--model-kind',kind,'--split-dir',str(SPLIT_DIR),'--output-dir',str(out),'--model-id',MODEL_ID,'--model-revision',MODEL_REVISION,'--tokenizer-id',TOKENIZER_ID,'--tokenizer-revision',TOKENIZER_REVISION,'--max-expanded-length',str(MAX_EXPANDED_LENGTH),'--max-byte-span-length',str(MAX_BYTE_SPAN_LENGTH),'--batch-size',str(MICROBATCH_SIZE),'--gradient-accumulation-steps',str(GRADIENT_ACCUMULATION_STEPS),'--seed',str(SEED),'--epochs',str(EPOCHS),'--learning-rate',str(LEARNING_RATE),'--weight-decay',str(WEIGHT_DECAY),'--warmup-ratio',str(WARMUP_RATIO),'--gradient-clip-norm',str(GRADIENT_CLIP_NORM),'--mixed-precision',REQUESTED_MIXED_PRECISION,'--validation-interval-steps',str(VALIDATION_INTERVAL_STEPS),'--num-workers',str(NUM_WORKERS),'--device',DEVICE_NAME]
def run_training(kind,out):
 ARTIFACT_ROOT.mkdir(parents=True,exist_ok=True); log_dir=ARTIFACT_ROOT/'logs'; log_dir.mkdir(exist_ok=True)
 log_path=log_dir/f'{kind}.log'
 with log_path.open('w',encoding='utf-8') as stream: subprocess.run(train_command(kind,out),cwd=REPO_ROOT,check=True,stdout=stream,stderr=subprocess.STDOUT,text=True)
 print('Training log:',log_path)
if RUN_FULL_BASELINE_TRAINING:
 ARTIFACT_ROOT.mkdir(parents=True,exist_ok=True)
 if BASELINE_OUTPUT.exists() and any(BASELINE_OUTPUT.iterdir()): raise FileExistsError('Choose a fresh output directory or explicit resume operation.')
 run_training('baseline',BASELINE_OUTPUT)
else: print('Full baseline training off; set RUN_FULL_BASELINE_TRAINING=True explicitly.')

## 10. Canonical training — explicit opt-in
Uses the same shared settings; architecture-specific differences remain. The comparison is whole-system only, not a byte-feature ablation.

In [ ]:
CANONICAL_OUTPUT=ARTIFACT_ROOT/'canonical'
if RUN_FULL_CANONICAL_TRAINING:
 ARTIFACT_ROOT.mkdir(parents=True,exist_ok=True)
 if CANONICAL_OUTPUT.exists() and any(CANONICAL_OUTPUT.iterdir()): raise FileExistsError('Choose a fresh output directory or explicit resume operation.')
 run_training('canonical',CANONICAL_OUTPUT)
else: print('Full canonical training off; set RUN_FULL_CANONICAL_TRAINING=True explicitly.')

## 11. Validation-only comparison manifest
The existing utility verifies shared settings and declares the comparison whole-system. No held-out split is read.

In [ ]:
from src.comparison import validate_comparison_checkpoints
from src.evaluation import evaluate_checkpoint_split
for kind,out in (('baseline',BASELINE_OUTPUT),('canonical',CANONICAL_OUTPUT)):
 pointer=out/'best_checkpoint.json'
 if pointer.is_file():
  info=json.loads(pointer.read_text()); checkpoint=out/info['checkpoint_dir']
  validation=evaluate_checkpoint_split(checkpoint,SPLIT_DIR,'validation',out/'validation_evaluation',device_name=DEVICE_NAME)
  print(kind+' validation:',json.dumps(validation,indent=2))
if (BASELINE_OUTPUT/'best_checkpoint.json').is_file() and (CANONICAL_OUTPUT/'best_checkpoint.json').is_file():
 bp=json.loads((BASELINE_OUTPUT/'best_checkpoint.json').read_text()); cp=json.loads((CANONICAL_OUTPUT/'best_checkpoint.json').read_text())
 report=validate_comparison_checkpoints(BASELINE_OUTPUT/bp['checkpoint_dir'],CANONICAL_OUTPUT/cp['checkpoint_dir'],SPLIT_DIR)
 print(json.dumps(report,indent=2,default=str)); assert report['comparison_type']=='whole-system; byte-feature effect is not isolated'
else: print('Comparison not run: both full training runs are required.')

## 12. Artifact export and recovery
Export full run bundles (weights, metadata, tokenizer assets, integrity data, resumable snapshots), configs, logs/history and provenance. Do not include source CSVs, credentials, caches or virtual environments. Kaggle working storage is not guaranteed to persist: download the archive or copy it to a user-managed store and verify it before ending the session.

In [ ]:
import shutil
ARTIFACT_ROOT.mkdir(parents=True,exist_ok=True)
(ARTIFACT_ROOT/'runtime_report.json').write_text(json.dumps(runtime_report,indent=2,default=str),encoding='utf-8')
(ARTIFACT_ROOT/'workflow_config.json').write_text(json.dumps({'shared':SHARED,'requested_precision':REQUESTED_MIXED_PRECISION,'effective_precision_probe':effective_precision,'model_revision':MODEL_REVISION,'tokenizer_revision':TOKENIZER_REVISION,'comparison':'whole-system only; byte-feature effect is not isolated'},indent=2),encoding='utf-8')
shutil.copy2(SPLIT_DIR/'manifest.json',ARTIFACT_ROOT/'dataset_manifest.json')
archive=shutil.make_archive(str(Path('/kaggle/working')/f'nlp-defense-m24-export-{SESSION_ID}'),'zip',root_dir=ARTIFACT_ROOT)
print({'archive':archive,'bytes':Path(archive).stat().st_size,'run_dirs':[p.name for p in ARTIFACT_ROOT.iterdir() if p.is_dir()],'resume':'Restore complete run directory; resume using checkpoints/latest_checkpoint.json or the checkpoint root.','persistence':'Download/export ZIP or copy to a user-managed store; Kaggle storage may be ephemeral.'})